# NHL Trap Games

Definition: A trap game is about a good team underperforming when they're expected to win

In [1]:
import sys
import os
import polars as pl
from tqdm import tqdm

base_dir = '../'
sys.path.insert(1, base_dir)

import src.config as cfg
import src.utils as utils
import src.sportsipy_utils as sportsipy_utils

## Pull Schedule for Each NHL Team

Note: We are using *All Situations* as a trap game is more about the actual results on the ice than it is about a team's true underlying ability

In [2]:
# Warning: NHL Schedules not loading - https://github.com/davidjkrause/sportsipy/issues/14
# sportsipy_utils.pull_nhl_schedule('TOR', 2024)

In [3]:
# https://www.naturalstattrick.com/games.php?fromseason=20242025&thruseason=20242025&stype=2&sit=all&loc=B&team=All&rate=n
schedule_folder = f'{base_dir}data/natural_stat_trick'

df_games = []
for filename in tqdm(os.listdir(schedule_folder)):
    if 'games_' in filename:
        schedule_filepath = os.path.join(schedule_folder, filename)
        df_games_filepath = pl.scan_csv(schedule_filepath, null_values=["-"]).collect()
        if not df_games_filepath.is_empty():
            df_games += [df_games_filepath]
    
df_games = (pl.concat(df_games)
            .unique(subset=['Game', 'Team'], keep='first')
            .with_columns(
                pl.col('Game').str.split(" - ").list.get(0).str.strip_chars().str.strptime(pl.Datetime,'%Y-%m-%d').alias('Date')
            )
            .select(['Game', 'Team', 'Date', 'GF', 'GA', 'GF%', 'xGF', 'xGA', 'xGF%', 'SH%', 'SV%' ,'PDO'])
           )
utils.logger.info(f"Loaded {df_games.shape[0]/2} unique games from natural stattrick") 
df_games.head(5)

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 84.58it/s]
2025-09-13 15:10:20,567 [1402038357.py:19] [INFO] Loaded 21528.0 unique games from natural stattrick


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64
""" 2019-10-30 - Panthers 4, Aval…","""Colorado Avalanche""",2019-10-30 00:00:00,3,4,42.86,3.12,3.15,49.75,10.0,89.47,0.995
""" 2023-01-19 - Panthers 6, Cana…","""Montreal Canadiens""",2023-01-19 00:00:00,2,6,25.0,2.86,6.53,30.48,8.0,85.0,0.93
""" 2019-12-09 - Islanders 5, Lig…","""Tampa Bay Lightning""",2019-12-09 00:00:00,1,5,16.67,2.72,2.06,56.97,3.13,79.17,0.823
""" 2007-12-13 - Avalanche 2, Pre…","""Colorado Avalanche""",2007-12-13 00:00:00,2,1,66.67,2.43,2.34,50.92,6.25,96.77,1.03
""" 2010-01-30 - Blackhawks 2, Hu…","""Chicago Blackhawks""",2010-01-30 00:00:00,2,4,33.33,4.54,2.87,61.24,4.88,83.33,0.882


In [5]:
df_missing = df_games.filter(pl.col("GF%").is_null())
utils.logger.info(f"Found {df_missing.shape[0]} / {df_games.shape[0]} rows without any goals in all situations")

2025-09-13 15:10:43,805 [659697710.py:2] [INFO] Found 104 / 43056 rows without any goals in all situations


## Identify Trap Games

In [6]:
strength_col = "Points"
strong_threshold = 1.1
weak_threshold = 0.9

df_games_processed = (
    df_games.filter(pl.col("GF%").is_not_null())
      .unique(subset=['Game', 'Team'], keep='first')
      .sort("Game", descending=False)
      .with_columns([
        pl.when(pl.col("Date").dt.month() >= 7).then(pl.col("Date").dt.year() + 1)
            .otherwise(pl.col("Date").dt.year())
            .alias("Season"),
          pl.when(pl.col("GF%")>50).then(pl.lit(2))
              .when(pl.col("GF%")<50).then(pl.lit(0))
              .otherwise(pl.lit(1))
          .alias("Points")
      ])
    .with_columns([
        pl.col(strength_col).cum_sum().over(["Team", "Season"]).alias("Season to Date Total incl Current"),
        pl.col("GF%").cum_count().over(["Team", "Season"]).alias("# Games Season to Date incl Current")
    ])
    .with_columns(
        ((pl.col("Season to Date Total incl Current") - pl.col(strength_col))/
         (pl.col("# Games Season to Date incl Current") - 1)).alias(f"Season to Date Average {strength_col}")
    )
    .with_columns([
        ((pl.col(f"Season to Date Average {strength_col}") > strong_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_strong"),
        ((pl.col(f"Season to Date Average {strength_col}") < weak_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_weak")
    ])
)  
                     
display(df_games_processed.tail(5))
df_games_processed.mean()

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool
""" 2025-04-17 - Islanders 1, Blu…","""New York Islanders""",2025-04-17 00:00:00,1,6,14.29,3.01,2.9,50.9,2.7,76.92,0.796,2025,0,71,82,0.876543,false,true
""" 2025-04-17 - Lightning 0, Ran…","""New York Rangers""",2025-04-17 00:00:00,4,0,100.0,2.22,2.17,50.52,18.18,100.0,1.182,2025,2,76,81,0.925,false,false
""" 2025-04-17 - Lightning 0, Ran…","""Tampa Bay Lightning""",2025-04-17 00:00:00,0,4,0.0,2.17,2.22,49.48,0.0,81.82,0.818,2025,0,95,82,1.17284,true,false
""" 2025-04-17 - Red Wings 3, Map…","""Detroit Red Wings""",2025-04-17 00:00:00,3,4,42.86,2.73,2.42,52.95,8.82,80.0,0.888,2025,0,75,82,0.925926,false,false
""" 2025-04-17 - Red Wings 3, Map…","""Toronto Maple Leafs""",2025-04-17 00:00:00,4,3,57.14,2.42,2.73,47.05,20.0,91.18,1.112,2025,2,105,82,1.271605,true,false


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak
str,str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,2016-08-21 05:52:29.413298,2.860286,2.860239,50.000698,2.768069,2.768026,50.000408,9.730112,90.27024,1.000004,2016.603511,1.000023,40.025028,40.005751,NaN,0.232981,0.225787


In [7]:
# Double check season derivation is reasonable given lockout in the 2012-2013 season and pandemic in 2019-2020/2020-2021 seasons
df_games_processed.group_by("Season").agg(pl.col("Game").count()).sort("Game").head(5)

Season,Game
i32,u32
2013,1434
2021,1730
2020,2160
2010,2448
2015,2448


In [8]:
df_opponent = df_games_processed.select([
    pl.col("Game"),
    pl.col("Team").alias("Opponent"),
    pl.col(f"Season to Date Average {strength_col}").alias(f"Opponent Season to Date Average {strength_col}"),
    pl.col("is_strong").alias("Opponent is_strong"),
    pl.col("is_weak").alias("Opponent is_weak")
])

df_games_with_opponent = (
    df_games_processed.join(df_opponent, on="Game", how="left")
    .filter(pl.col("Team") != pl.col("Opponent"))
    .with_columns(
        (pl.col(f"Season to Date Average {strength_col}") - pl.col(f"Opponent Season to Date Average {strength_col}")).alias(f"Delta {strength_col}")
    )

    # Tag previous opponent
    .sort(["Team", "Date"])
    .with_columns([
        pl.col(col).shift(1).over(["Team", "Season"]).alias(f"Previous {col}")
        for col in ["Opponent", "Opponent is_strong", "Opponent is_weak", "GF%"]
    ])

    # Tag Trap Games: Strong Team that beat another strong team in the previous game faces a weak opponent in the next game
    .with_columns(
        pl.when((pl.col("is_strong"))&(pl.col("Opponent is_weak"))
                &(pl.col("Previous Opponent is_strong"))&(pl.col("Previous GF%") > 50))
            .then(pl.lit(True))
        .otherwise(pl.lit(False)).alias("Trap Game")
    )
)

assert df_games_with_opponent.shape[0] == df_games_processed.shape[0]

df_games_with_opponent.filter(pl.col("Trap Game"))

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak,Opponent,Opponent Season to Date Average Points,Opponent is_strong,Opponent is_weak,Delta Points,Previous Opponent,Previous Opponent is_strong,Previous Opponent is_weak,Previous GF%,Trap Game
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str,f64,bool,bool,f64,str,bool,bool,f64,bool
""" 2013-03-31 - Ducks 1, Blue Ja…","""Anaheim Ducks""",2013-03-31 00:00:00,1,2,33.33,1.8,1.46,55.23,3.33,88.89,0.922,2013,0,44,35,1.294118,true,false,"""Columbus Blue Jackets""",0.787879,false,true,0.506239,"""Chicago Blackhawks""",true,false,66.67,true
""" 2013-04-27 - Coyotes 5, Ducks…","""Anaheim Ducks""",2013-04-27 00:00:00,3,5,37.5,4.23,3.13,57.45,6.52,83.87,0.904,2013,0,57,48,1.212766,true,false,"""Phoenix Coyotes""",0.888889,false,true,0.323877,"""Vancouver Canucks""",true,false,75.0,true
""" 2013-12-09 - Islanders 2, Duc…","""Anaheim Ducks""",2013-12-09 00:00:00,5,2,71.43,3.51,3.57,49.62,16.13,94.59,1.107,2014,2,44,33,1.3125,true,false,"""New York Islanders""",0.5,false,true,0.8125,"""St Louis Blues""",true,false,71.43,true
""" 2013-12-15 - Oilers 2, Ducks …","""Anaheim Ducks""",2013-12-15 00:00:00,3,2,60.0,2.59,1.6,61.77,8.82,92.0,1.008,2014,2,48,35,1.352941,true,false,"""Edmonton Oilers""",0.617647,false,true,0.735294,"""Minnesota Wild""",true,false,66.67,true
""" 2014-01-03 - Oilers 2, Ducks …","""Anaheim Ducks""",2014-01-03 00:00:00,5,2,71.43,4.49,1.6,73.73,13.51,88.89,1.024,2014,2,62,43,1.428571,true,false,"""Edmonton Oilers""",0.604651,false,true,0.82392,"""San Jose Sharks""",true,false,66.67,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
""" 2023-12-27 - Jets 1, Blackhaw…","""Winnipeg Jets""",2023-12-27 00:00:00,1,2,33.33,3.96,1.6,71.21,2.63,92.0,0.946,2024,0,41,33,1.28125,true,false,"""Chicago Blackhawks""",0.636364,false,true,0.644886,"""Boston Bruins""",true,false,83.33,true
""" 2024-04-16 - Kraken 3, Jets 4""","""Winnipeg Jets""",2024-04-16 00:00:00,4,3,57.14,2.96,1.69,63.67,16.67,86.96,1.036,2024,2,103,81,1.2625,true,false,"""Seattle Kraken""",0.8375,false,true,0.425,"""Colorado Avalanche""",true,false,100.0,true
""" 2025-03-16 - Jets 3, Kraken 2""","""Winnipeg Jets""",2025-03-16 00:00:00,3,2,60.0,3.23,2.79,53.66,11.54,90.48,1.02,2025,2,94,68,1.373134,true,false,"""Seattle Kraken""",0.895522,false,true,0.477612,"""Dallas Stars""",true,false,80.0,true


In [9]:
df_leafs_sample = df_games_with_opponent.filter((pl.col("Team")=="Toronto Maple Leafs")&(pl.col("Season")==2025))
display(df_leafs_sample.head(5))

df_leafs_sample.write_csv(f"{base_dir}data/derived/Leafs Trap Games Sample.csv")

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average Points,is_strong,is_weak,Opponent,Opponent Season to Date Average Points,Opponent is_strong,Opponent is_weak,Delta Points,Previous Opponent,Previous Opponent is_strong,Previous Opponent is_weak,Previous GF%,Trap Game
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str,f64,bool,bool,f64,str,bool,bool,f64,bool
""" 2024-10-09 - Maple Leafs 0, C…","""Toronto Maple Leafs""",2024-10-09 00:00:00,0,1,0.0,6.09,2.97,67.2,0.0,96.3,0.963,2025,0,0,1,NaN,false,false,"""Montreal Canadiens""",NaN,false,false,NaN,null,null,null,null,false
""" 2024-10-10 - Maple Leafs 4, D…","""Toronto Maple Leafs""",2024-10-10 00:00:00,4,2,66.67,2.11,2.62,44.67,18.18,91.3,1.095,2025,2,2,2,0.0,false,false,"""New Jersey Devils""",2.0,false,false,-2.0,"""Montreal Canadiens""",false,false,0.0,false
""" 2024-10-12 - Penguins 2, Mapl…","""Toronto Maple Leafs""",2024-10-12 00:00:00,4,2,66.67,2.95,1.89,60.94,12.12,91.3,1.034,2025,2,4,3,1.0,false,false,"""Pittsburgh Penguins""",1.0,false,false,0.0,"""New Jersey Devils""",false,false,66.67,false
""" 2024-10-16 - Kings 2, Maple L…","""Toronto Maple Leafs""",2024-10-16 00:00:00,6,2,75.0,1.86,4.75,28.12,23.08,94.12,1.172,2025,2,6,4,1.333333,false,false,"""Los Angeles Kings""",0.666667,false,false,0.666667,"""Pittsburgh Penguins""",false,false,66.67,false
""" 2024-10-19 - Rangers 4, Maple…","""Toronto Maple Leafs""",2024-10-19 00:00:00,1,4,20.0,3.98,5.07,44.0,2.86,86.21,0.891,2025,0,6,5,1.5,false,false,"""New York Rangers""",1.5,false,false,0.0,"""Los Angeles Kings""",false,false,75.0,false


## Aggregate Tables

In [22]:
from typing import List

In [92]:
def get_agg_tables(df: pl.DataFrame, groupby_cols: List[str]) -> [pl.DataFrame, pl.DataFrame]:
    df_agg = (
        df.filter((pl.col("is_strong"))&(pl.col("Opponent is_weak"))) # compare strong vs weak games
        .group_by(groupby_cols + ['Trap Game']) # depending on whether the strong team won against another strong team the previous game
        .agg([pl.col('Game').count().alias('Sample Size').cast(pl.Int32)] 
             + [pl.col(col).mean() for col in ['GF', 'GA', 'GF%', 'xGF', 'xGA', 'xGF%', 'SH%', 'SV%', 'PDO']])
        .sort(groupby_cols + ['Trap Game'])
    )

    # Calculate % change
    value_cols = [col for col in df_agg.columns if col not in groupby_cols and col != 'Trap Game']
    if len(groupby_cols) == 0:
        df_pct_change = df_agg.filter(pl.col('Trap Game')).select(value_cols) / df_agg.filter(~pl.col('Trap Game')).select(value_cols) - 1
    else:
        df_pct_change = (df_agg
            .pivot(index=groupby_cols, on='Trap Game', values=value_cols)
            .with_columns([(pl.col(f'{col}_true') / pl.col(f'{col}_false') - 1).alias(col) 
                           for col in value_cols])
            .drop([f'{col}_true' for col in value_cols] + [f'{col}_false' for col in value_cols])
        )
    
    return df_agg, df_pct_change

In [98]:
for groupby_cols in [[], ['Season'], ['Team']]:
    df_agg, df_pct_change = get_agg_tables(df_games_with_opponent, groupby_cols)

    utils.logger.info(f"Pivot tables by {groupby_cols}")
    display(df_agg)
    display(df_pct_change)

2025-09-13 15:45:55,530 [2454748982.py:4] [INFO] Pivot tables by []


Trap Game,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
bool,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
false,2673,3.27572,2.575383,56.401575,3.03015,2.574598,54.043471,10.447995,90.691938,1.011404
true,482,3.475104,2.522822,58.537116,3.079025,2.51471,54.922676,11.067282,90.868071,1.019342


Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.819678,0.060867,-0.020409,0.037863,0.01613,-0.023261,0.016268,0.059273,0.001942,0.007849


2025-09-13 15:45:55,550 [2454748982.py:4] [INFO] Pivot tables by ['Season']


Season,Trap Game,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
i32,bool,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
2008,false,79,2.962025,2.78481,50.678481,2.750759,2.531899,51.641667,9.945696,89.458101,0.994013
2008,true,11,2.272727,2.727273,47.910909,2.377273,2.783636,48.54,8.218182,91.112727,0.993273
2009,false,132,3.280303,2.818182,52.811364,2.91197,2.431364,54.231615,10.397955,89.906439,1.00303
2009,true,24,3.583333,2.291667,59.783333,3.018333,2.452917,55.09,10.694167,92.03375,1.027292
2010,false,153,3.202614,2.54902,55.710196,2.912157,2.641176,52.853595,10.25902,90.693203,1.009549
…,…,…,…,…,…,…,…,…,…,…,…
2023,true,40,4.175,2.725,61.58425,3.569,2.7445,56.479,13.00225,89.90125,1.02905
2024,false,163,3.723926,2.564417,60.89816,3.407055,2.769571,55.17227,11.577975,90.44681,1.020282
2024,true,24,3.333333,2.5,54.065,3.600833,2.2325,61.899167,10.362083,89.84125,1.002125


Season,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2008,-0.860759,-0.232712,-0.020661,-0.05461,-0.135776,0.099426,-0.060061,-0.173695,0.018496,-0.000744
2009,-0.818182,0.092379,-0.186828,0.132016,0.036526,0.008865,0.015828,0.028488,0.023661,0.024188
2010,-0.836601,0.011673,0.192615,-0.101436,0.042114,0.089969,-0.022507,-0.006182,-0.010184,-0.009855
2011,-0.869919,-0.04986,-0.143113,-0.036872,-0.118851,-0.135288,0.011585,0.105414,0.00807,0.017185
2012,-0.836207,0.002882,0.140776,-0.051503,-0.077285,-0.051026,-0.011239,0.027859,-0.011792,-0.008041
…,…,…,…,…,…,…,…,…,…,…
2021,-0.896373,-0.153253,-0.039089,-0.077902,-0.086248,-0.014759,-0.029715,-0.165567,0.012444,-0.006501
2022,-0.666667,0.157025,0.026616,0.041341,0.052369,0.009371,0.016959,0.15304,-0.001347,0.015106
2023,-0.761905,0.092523,0.087411,0.01103,-0.007413,0.042852,-0.022642,0.126671,-0.005731,0.009303


2025-09-13 15:45:55,594 [2454748982.py:4] [INFO] Pivot tables by ['Team']


Team,Trap Game,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,bool,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Anaheim Ducks""",false,47,3.170213,2.638298,55.260213,2.885106,2.484043,53.505745,10.39766,90.533404,1.00934
"""Anaheim Ducks""",true,10,2.5,2.6,48.869,2.942,2.398,54.578,7.33,89.976,0.973
"""Atlanta Thrashers""",false,7,3.285714,3.0,49.03,2.44,3.198571,44.135714,13.108571,89.771429,1.028857
"""Boston Bruins""",false,167,3.281437,2.538922,57.247006,2.889341,2.492754,53.878144,10.168743,90.965808,1.011353
"""Boston Bruins""",true,35,3.342857,1.657143,66.433714,3.060857,2.493143,55.273429,10.008286,94.050571,1.0406
…,…,…,…,…,…,…,…,…,…,…,…
"""Vegas Golden Knights""",true,18,2.833333,2.944444,46.891667,2.797778,2.595556,51.740556,10.036667,89.192222,0.992222
"""Washington Capitals""",false,165,3.248485,2.848485,52.756485,3.101879,2.82,52.225515,10.612061,89.957333,1.005709
"""Washington Capitals""",true,30,3.766667,2.333333,62.531,2.907333,2.693667,51.794667,12.459667,92.197,1.046533


Team,Sample Size,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Anaheim Ducks""",-0.787234,-0.211409,-0.014516,-0.115657,0.01972,-0.034638,0.02004,-0.295034,-0.006157,-0.036004
"""Atlanta Thrashers""",null,null,null,null,null,null,null,null,null,null
"""Boston Bruins""",-0.790419,0.018717,-0.347305,0.160475,0.059362,0.000156,0.025897,-0.015779,0.033911,0.028918
"""Buffalo Sabres""",-0.92,0.029412,0.008065,0.036285,-0.101934,0.223984,-0.172712,0.129079,0.014869,0.027668
"""Calgary Flames""",-0.796296,0.029712,-0.217391,0.149074,0.031569,0.028712,0.006682,-0.014185,0.023569,0.019165
…,…,…,…,…,…,…,…,…,…,…
"""Toronto Maple Leafs""",-0.785714,0.031826,-0.11976,0.092157,-0.022716,-0.028253,0.008986,0.063838,0.013034,0.0186
"""Vancouver Canucks""",-0.918367,-0.222222,-0.001073,-0.02217,-0.110338,-0.06626,-0.011934,-0.255652,0.006038,-0.021698
"""Vegas Golden Knights""",-0.772152,-0.21462,0.145868,-0.190139,-0.147743,-0.037194,-0.052637,-0.080678,-0.015965,-0.023014
